# Hard-adversarial ranking check — Dbpedia

For each held-out anchor, does the model score `cos(anchor, positive)` HIGHER than `cos(anchor, negative)`, where the positive is answer-equivalent but lexically divergent, and the negative is lexically similar but answer-different?

A lexical-only encoder should get this backwards; a result-aware encoder (SparqlCE) should get it right. Run once on all rows, then again filtered to `meets_strict_bar: true` only — that's the headline number (see `datasets/README.md` for how few rows actually clear both thresholds).

In [1]:
import sys, os
from pathlib import Path

_PROJECT_ROOT = Path.cwd().parents[2]
if str(_PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(_PROJECT_ROOT))

_RQ1_DIR = Path.cwd().parent
if str(_RQ1_DIR) not in sys.path:
    sys.path.insert(0, str(_RQ1_DIR))

from evaluate_hard_adversarial import load_hard_testset, evaluate_all
print(f"Project root: {_PROJECT_ROOT}")

TEST_DATA_PATH = os.path.join(_PROJECT_ROOT, "datasets", "dbpedia_testset_RQ1_hard.json")
CHECKPOINT_PATH = os.path.join(_PROJECT_ROOT, "checkpoints", "sparqlce_dbpedia", "best_model.pt")
E5_FT_PATH = os.path.join(_PROJECT_ROOT, "checkpoints", "e5_square_dbpedia", "best_model")
BATCH = 32


/upb/users/g/ganeshk/profiles/unix/cs/.conda/envs/sparql/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Project root: /upb/users/g/ganeshk/profiles/unix/cs/SPARQL-Embeddings


## Full set (all rows, includes rows below the strict bar — noisier, larger-N check)

In [2]:
rows_all = load_hard_testset(TEST_DATA_PATH, strict_only=False)
results_all = evaluate_all(rows_all, CHECKPOINT_PATH, e5_ft_path=E5_FT_PATH, batch_size=BATCH, label="(full) ")



Hard-adversarial ranking check (full) (n=500)
Loading model from /upb/users/g/ganeshk/profiles/unix/cs/SPARQL-Embeddings/checkpoints/sparqlce_dbpedia/best_model.pt...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 8234.10it/s]


SPARQLCE                        ranking-acc=0.920  mean_margin(pos-neg)=+0.5240  n=500
TF-IDF                          ranking-acc=0.356  mean_margin(pos-neg)=-0.0691  n=500
BM25                            ranking-acc=0.416  mean_margin(pos-neg)=+1.3278  n=500


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 8869.64it/s]


SBERT/MiniLM                    ranking-acc=0.562  mean_margin(pos-neg)=-0.0183  n=500


...

SPBERT (zero-shot)              ranking-acc=0.510  mean_margin(pos-neg)=-0.0043  n=500


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 290/290 [00:00<00:00, 9687.31it/s]


Qwen2.5-Coder (zero-shot)       ranking-acc=0.492  mean_margin(pos-neg)=-0.0034  n=500


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 45672.59it/s]


E5-base-v2 (fine-tuned)         ranking-acc=0.968  mean_margin(pos-neg)=+0.6008  n=500


## Strict-bar rows only (the headline number — see `meets_strict_bar` in the README)

In [3]:
rows_strict = load_hard_testset(TEST_DATA_PATH, strict_only=True)
results_strict = evaluate_all(rows_strict, CHECKPOINT_PATH, e5_ft_path=E5_FT_PATH, batch_size=BATCH, label="(strict-bar) ")



Hard-adversarial ranking check (strict-bar) (n=87)
Loading model from /upb/users/g/ganeshk/profiles/unix/cs/SPARQL-Embeddings/checkpoints/sparqlce_dbpedia/best_model.pt...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 9194.79it/s]


SPARQLCE                        ranking-acc=0.885  mean_margin(pos-neg)=+0.4475  n=87
TF-IDF                          ranking-acc=0.000  mean_margin(pos-neg)=-0.6190  n=87
BM25                            ranking-acc=0.000  mean_margin(pos-neg)=-10.5523  n=87


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 8407.87it/s]


SBERT/MiniLM                    ranking-acc=0.000  mean_margin(pos-neg)=-0.2240  n=87


...

SPBERT (zero-shot)              ranking-acc=0.000  mean_margin(pos-neg)=-0.0350  n=87


Loading weights: 100%|██████████| 290/290 [00:00<00:00, 5782.94it/s]


Qwen2.5-Coder (zero-shot)       ranking-acc=0.000  mean_margin(pos-neg)=-0.0221  n=87


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 8518.92it/s]


E5-base-v2 (fine-tuned)         ranking-acc=0.885  mean_margin(pos-neg)=+0.4272  n=87
